In [ ]:
#@title Installing Java version 17 due to Neo4j requirements
!apt-get install openjdk-17-jre-headless -qq > /dev/null
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
!update-alternatives --set java /usr/lib/jvm/java-17-openjdk-amd64/bin/java
!java -version

openjdk version "17.0.12" 2024-07-16
OpenJDK Runtime Environment (build 17.0.12+7-Ubuntu-1ubuntu222.04)
OpenJDK 64-Bit Server VM (build 17.0.12+7-Ubuntu-1ubuntu222.04, mixed mode, sharing)


In [ ]:
#@title Installing Neo4J server on Google colab
!wget -q https://neo4j.com/artifact.php?name=neo4j-community-5.21.0-unix.tar.gz -O neo4j.tar.gz
# decompress and rename
!tar -xf neo4j.tar.gz  # or --strip-components=1
!mv neo4j-community-5.21.0 nj
# download apoc plugins and enable apoc command
!wget -q https://github.com/neo4j/apoc/releases/download/5.21.0/apoc-5.21.0-core.jar -O nj/plugins/apoc-5.21.0-core.jar
!wget -q https://github.com/neo4j/graph-data-science/releases/download/2.8.0/neo4j-graph-data-science-2.8.0.jar -O nj/plugins/gds-2.8.0.jar
!echo "dbms.security.procedures.unrestricted=gds.*,apoc.*,apoc.text.*" >> nj/conf/neo4j.conf
!echo "dbms.security.procedures.allowlist=gds.*,apoc.*,apoc.text.*" >> nj/conf/neo4j.conf
# disable password, and start server
!sed -i '/#dbms.security.auth_enabled/s/^#//g' nj/conf/neo4j.conf

In [ ]:
!mv /content/neo4j_job_descriptions.dump /content/neo4j.dump

In [ ]:
!du -sh /content/neo4j.dump

20M	/content/neo4j.dump


In [ ]:
!nj/bin/neo4j-admin database load neo4j --from-path=/content/ --overwrite-destination=true

Done: 43 files, 271.3MiB processed.


In [ ]:
!nj/bin/neo4j start

Directories in use:
home:         /content/nj
config:       /content/nj/conf
logs:         /content/nj/logs
plugins:      /content/nj/plugins
import:       /content/nj/import
data:         /content/nj/data
certificates: /content/nj/certificates
licenses:     /content/nj/licenses
run:          /content/nj/run
Starting Neo4j.
Started neo4j (pid:1605). It is available at http://localhost:7474
There may be a short delay until the server is ready.


In [ ]:
!pip install py2neo openai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.2/177.2 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.7/336.7 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 3.3 MB/s eta 0:00:00


In [ ]:
from py2neo import Graph
graph = Graph("bolt://localhost:7687")

In [ ]:
def run_cypher_in_neo4j(cypher_query: str,df=False) -> str:
    result = graph.run(cypher_query).to_data_frame()
    return result if df else result.to_markdown(index=False)

In [ ]:
nodes_and_relations = run_cypher_in_neo4j("""CALL db.labels() YIELD label
RETURN "Node" AS Type, label AS Name
UNION ALL
CALL db.relationshipTypes() YIELD relationshipType
RETURN "Relationship" AS Type, relationshipType AS Name""")
print(nodes_and_relations)

| Type         | Name                 |
|:-------------|:---------------------|
| Node         | Chunk                |
| Node         | __Entity__           |
| Node         | CANDIDATE            |
| Node         | SOFT_SKILL           |
| Node         | HARD_SKILL           |
| Node         | PROFESSION           |
| Node         | EDUCATION            |
| Node         | JOB_DESCRIPTION      |
| Node         | COURSE               |
| Node         | ORGANIZATION         |
| Node         | PROJECT              |
| Node         | REFERENCE            |
| Relationship | MENTIONS             |
| Relationship | HAS_SOFT_SKILL       |
| Relationship | HAS_HARD_SKILL       |
| Relationship | HAS_PROFESSION       |
| Relationship | HAS_EDUCATION        |
| Relationship | REQUIRES_SKILL       |
| Relationship | COMPLETED_COURSE     |
| Relationship | ATTENDED_INSTITUTION |
| Relationship | WORKED_ON            |
| Relationship | HAS_RESPONSIBILITY   |
| Relationship | WORKS_AT             |


In [ ]:
node_relations_map = run_cypher_in_neo4j("""MATCH (n)-[r]->()
WITH labels(n) AS NodeLabels, collect(DISTINCT type(r)) AS Relationships
RETURN DISTINCT
  NodeLabels[-1] AS Node,
  Relationships
""")
print(node_relations_map)

| Node            | Relationships                                                                                                                                                 |
|:----------------|:--------------------------------------------------------------------------------------------------------------------------------------------------------------|
| Chunk           | ['MENTIONS']                                                                                                                                                  |
| CANDIDATE       | ['HAS_SOFT_SKILL', 'HAS_HARD_SKILL', 'HAS_PROFESSION', 'HAS_EDUCATION', 'COMPLETED_COURSE', 'WORKED_ON', 'ATTENDED_INSTITUTION', 'WORKS_AT', 'HAS_REFERENCE'] |
| JOB_DESCRIPTION | ['REQUIRES_SKILL', 'REQUIRES_EDUCATION']                                                                                                                      |
| COURSE          | ['ATTENDED_INSTITUTION']                                                        

In [ ]:
run_cypher_in_neo4j("MATCH (n:CANDIDATE) RETURN COUNT(n) AS candidates_counts")

'|   candidates_counts |\n|--------------------:|\n|                 108 |'

In [ ]:
# get the id from random Chunk node
result = run_cypher_in_neo4j("""
MATCH (c:Chunk)
WITH c, rand() AS random
ORDER BY random
LIMIT 1
RETURN id(c) AS _id, c.text AS text
""",df=True)
id = result.loc[0,'_id']
result

,_id,text
0,17,Akash N. Langade akashlangade05@gmail.com: {'s...


In [ ]:
# retrieve text using Chunk node ID
run_cypher_in_neo4j(f"""
MATCH (c:Chunk)-[:MENTIONS]->(n:CANDIDATE)
WHERE id(c) = {id}
RETURN id(c), c.text""",df=True)

,id(c),c.text
0,17,Akash N. Langade akashlangade05@gmail.com: {'s...


In [ ]:
# Can add system prompting to guide the model to call functions and perform in specific ways
system_prompt = f"""
Assistant is a helpful assistant that helps users get answers to questions about a Neo4jGraph.
Assistant has access to several tools and sometimes you may need to call multiple tools in sequence to get answers for your users.

The Neo4j graph schema, including all entities and relationships available, is as follows (use the exact names):
{nodes_and_relations}

Here is the map of nodes and relationships:
{node_relations_map}
"""
next_messages = [
    {
        "role": "system",
        "content": system_prompt,
    }]

In [ ]:
tool_run_cypher_in_neo4j = {
    "type": "function",
    "function": {
        "name": "run_cypher_in_neo4j",
        "description": "Execute a Cypher query in a Neo4j database and return the result as a Markdown string.",
        "parameters": {
            "type": "object",
            "properties": {
                "cypher_query": {
                    "type": "string",
                    "description":
"""The Cypher query to execute always has alias in the result and
Always include the following fun to avoid retrieve 'embeddings' field:
WITH apoc.map.removeKey(properties(...), 'embedding') AS ..
e.g. 'MATCH (n:...) RETURN COUNT(n) AS ...'.""",
                },
            },
            "required": ["cypher_query"],
        },
    },
}


In [ ]:
from openai import AzureOpenAI
from google.colab import userdata
import json
from pprint import pprint

client = AzureOpenAI(
    api_key=userdata.get('AZURE_API_KEY'),
    api_version=userdata.get('AZURE_API_VERSION'),
    azure_endpoint=userdata.get('AZURE_BASE_URL')
)

import concurrent.futures

BLUE,GREEN,MANGETA,RESET = '\033[94m', '\033[92m', '\033[95m', '\033[0m'
def dict_to_expression_string(dictionary):
    return ' '.join(f'{BLUE}{key}{RESET}={GREEN}{value}{RESET}' for key, value in dictionary.items())

def run_multiturn_conversation(messages, tools, available_functions):
    response = client.chat.completions.create(
        messages=messages,
        tools=tools,
        tool_choice="auto",
        model=userdata.get('AZURE_MODEL_NAME'),
        temperature=1e-4,
    )

    while response.choices[0].finish_reason == "tool_calls":
        response_message = response.choices[0].message

        # print("Assistant Response:")
        # print(response_message)
        with concurrent.futures.ThreadPoolExecutor() as executor:
            futures, names, args = [], [], []

            for tool in response_message.tool_calls:
                print("Invoking Recommended Function call:")
                print(tool.function)
                print()

                function_name = tool.function.name

                names.append(function_name)
                function_to_call = available_functions[function_name]
                function_args = json.loads(tool.function.arguments)
                args.append(function_args)
                futures.append(executor.submit(function_to_call,**function_args))

        for future, name, arg in zip(futures, names, args):
            function_response = future.result()


            print(f"Output of function call of {MANGETA}{name}{RESET} with {dict_to_expression_string(arg)}:")
            print(function_response)
            print()

            messages.append(
                {
                    "role": response_message.role,
                    "function_call": {
                        "name": tool.function.name,
                        "arguments": tool.function.arguments,
                    },
                    "content": None,
                }
            )
            messages.append(
                {
                    "role": "function",
                    "name": function_name,
                    "content": function_response,
                }
            )

        print("Messages in next request:")
        for message in messages:
            print(message)
        print()

        response = client.chat.completions.create(
            messages=messages,
            tools=tools,
            tool_choice="auto",
            model=userdata.get('AZURE_MODEL_NAME'),
            temperature=1e-4,
        )
        # ends with


    messages.append(
        {
            "role": response.choices[0].message.role,
            "content": response.choices[0].message.content,
        }
    )  # extend conversation with assistant response

    return response

In [ ]:
tools = [
    tool_run_cypher_in_neo4j,
]

available_functions = {
    "run_cypher_in_neo4j": run_cypher_in_neo4j,
}

In [ ]:
#3seconds
from IPython.display import Markdown

next_messages = [next_messages[0]]
next_messages.append({
        "role": "user",
        "content": "list 2 candidates",
    })

assistant_response = run_multiturn_conversation(
    next_messages, tools, available_functions
)
print("Final Response:")
print(assistant_response.choices[0].message)
print("Conversation complete!")
Markdown(next_messages[-1]['content'])

Invoking Recommended Function call:
Function(arguments='{"cypher_query":"MATCH (c:CANDIDATE) WITH apoc.map.removeKey(properties(c), \'embedding\') AS candidate RETURN candidate LIMIT 2"}', name='run_cypher_in_neo4j')

Output of function call of run_cypher_in_neo4j with cypher_query=MATCH (c:CANDIDATE) WITH apoc.map.removeKey(properties(c), 'embedding') AS candidate RETURN candidate LIMIT 2:
| candidate                                                                                                               |
|:------------------------------------------------------------------------------------------------------------------------|
| {'id': 'MUHAMMAD. ISAK ALI', 'name': 'MUHAMMAD. ISAK ALI', 'triplet_source_id': '9fef5777-4f37-4253-a50d-79f5e01b9cbc'} |
| {'id': 'Arif Mallick', 'name': 'Arif Mallick', 'triplet_source_id': 'a148c7ba-7fba-4513-83ef-324e3c5eda91'}             |

Messages in next request:
{'role': 'system', 'content': "\nAssistant is a helpful assistant that helps users 

Here are 2 candidates from the database:

1. **MUHAMMAD. ISAK ALI**
   - ID: MUHAMMAD. ISAK ALI
   - Triplet Source ID: 9fef5777-4f37-4253-a50d-79f5e01b9cbc

2. **Arif Mallick**
   - ID: Arif Mallick
   - Triplet Source ID: a148c7ba-7fba-4513-83ef-324e3c5eda91

In [ ]:
#6seconds
next_messages = [next_messages[0]]
next_messages.append({
        "role": "user",
        "content": "How many skills exists?",
    })

assistant_response = run_multiturn_conversation(
    next_messages, tools, available_functions
)
print("Final Response:")
Markdown(next_messages[-1]['content'])

Invoking Recommended Function call:
Function(arguments='{"cypher_query": "MATCH (n:SOFT_SKILL) RETURN COUNT(n) AS soft_skill_count"}', name='run_cypher_in_neo4j')

Invoking Recommended Function call:
Function(arguments='{"cypher_query": "MATCH (n:HARD_SKILL) RETURN COUNT(n) AS hard_skill_count"}', name='run_cypher_in_neo4j')

Output of function call of run_cypher_in_neo4j with cypher_query=MATCH (n:SOFT_SKILL) RETURN COUNT(n) AS soft_skill_count:
|   soft_skill_count |
|-------------------:|
|                201 |

Output of function call of run_cypher_in_neo4j with cypher_query=MATCH (n:HARD_SKILL) RETURN COUNT(n) AS hard_skill_count:
|   hard_skill_count |
|-------------------:|
|                301 |

Messages in next request:
{'role': 'system', 'content': "\nAssistant is a helpful assistant that helps users get answers to questions about a Neo4jGraph.\nAssistant has access to several tools and sometimes you may need to call multiple tools in sequence to get answers for your users.\

There are a total of 502 skills in the database, comprising 201 soft skills and 301 hard skills.

In [ ]:
#10seconds
next_messages = [next_messages[0]]
next_messages.append({
        "role": "user",
        "content": "list 3 candidates with their skills",
    })

assistant_response = run_multiturn_conversation(
    next_messages, tools, available_functions
)
print("Final Response:")
Markdown(next_messages[-1]['content'])

Invoking Recommended Function call:
Function(arguments='{"cypher_query":"MATCH (c:CANDIDATE)-[:HAS_SOFT_SKILL|HAS_HARD_SKILL]->(s) WITH c, COLLECT(s.name) AS skills RETURN c.name AS candidate, skills LIMIT 3"}', name='run_cypher_in_neo4j')

Output of function call of run_cypher_in_neo4j with cypher_query=MATCH (c:CANDIDATE)-[:HAS_SOFT_SKILL|HAS_HARD_SKILL]->(s) WITH c, COLLECT(s.name) AS skills RETURN c.name AS candidate, skills LIMIT 3:
| candidate          | skills                                                                                                                                                                                                                                                                                                                                                                                                                           |
|:-------------------|:---------------------------------------------------------------------------------------------

Here are 3 candidates along with their skills:

1. **MUHAMMAD. ISAK ALI**
   - Skills: Being organized, Problem-solving, Report writing, Administrative support, Positive, Innovative, Able to establish good working relationships, Document management, Correspondence, Approachable

2. **Arif Mallick**
   - Skills: Ability to interact and effectively communicate with people from diverse backgrounds, highlighting teamwork and problem solving, Receiving of Direct, Very keen eye for detail, Exceptional communication, Excellent communication, interpersonal and leadership skills, Strong analytical abilities in inventory situations, Creation of new data, Assisted in developing, Huge knowledge, Finding urgent

3. **A.R.MOHAMED ITHRIS**
   - Skills: Problem-solving skills, Written and communication skills, Strong analytical skills, Comprehensive problem solving, PPAP, 7QC tools, Project Management, FMEA, supplier selection, APQP

In [ ]:
#19seconds
next_messages = [next_messages[0]]
next_messages.append({
        "role": "user",
        "content": "list 3 candidates with their soft and hard skills",
    })

assistant_response = run_multiturn_conversation(
    next_messages, tools, available_functions
)
print("Final Response:")
Markdown(next_messages[-1]['content'])

Invoking Recommended Function call:
Function(arguments='{"cypher_query":"MATCH (c:CANDIDATE)-[:HAS_SOFT_SKILL]->(ss:SOFT_SKILL), (c)-[:HAS_HARD_SKILL]->(hs:HARD_SKILL) WITH c, COLLECT(DISTINCT ss.name) AS soft_skills, COLLECT(DISTINCT hs.name) AS hard_skills RETURN c.name AS candidate_name, soft_skills, hard_skills LIMIT 3"}', name='run_cypher_in_neo4j')

Output of function call of run_cypher_in_neo4j with cypher_query=MATCH (c:CANDIDATE)-[:HAS_SOFT_SKILL]->(ss:SOFT_SKILL), (c)-[:HAS_HARD_SKILL]->(hs:HARD_SKILL) WITH c, COLLECT(DISTINCT ss.name) AS soft_skills, COLLECT(DISTINCT hs.name) AS hard_skills RETURN c.name AS candidate_name, soft_skills, hard_skills LIMIT 3:
| candidate_name     | soft_skills                                                                                                                                                                                                                                                                            | hard_skills         

Here are 3 candidates along with their soft and hard skills:

1. **MUHAMMAD. ISAK ALI**
   - **Soft Skills**: 
     - Being organized
     - Problem-solving
     - Positive
     - Innovative
     - Able to establish good working relationships
     - Approachable
   - **Hard Skills**: 
     - Report writing
     - Administrative support
     - Document management
     - Correspondence

2. **Arif Mallick**
   - **Soft Skills**: 
     - Ability to interact and effectively communicate with people from diverse backgrounds, highlighting teamwork and problem solving
     - Very keen eye for detail
     - Excellent communication, interpersonal and leadership skills
     - Strong analytical abilities in inventory situations
   - **Hard Skills**: 
     - Receiving of Direct
     - Exceptional communication
     - Creation of new data
     - Assisted in developing
     - Huge knowledge
     - Finding urgent

3. **Vijay Jawre**
   - **Soft Skills**: 
     - Comprehensive problem solving
     - Hard Working
     - Ability to maintain calm under pressure
     - Willing to take on new responsibilities and respond to it
   - **Hard Skills**: 
     - WAN
     - LAN
     - Server health
     - Antivirus server
     - Active Directory
     - Hardware faults

In [ ]:
#36seconds
next_messages = [next_messages[0]]
next_messages.append({
        "role": "user",
        "content": "generate an analysis of the whole database(math measures)",
    })

assistant_response = run_multiturn_conversation(
    next_messages, tools, available_functions
)
print("Final Response:")
Markdown(next_messages[-1]['content'])

Invoking Recommended Function call:
Function(arguments='{"cypher_query": "MATCH (n) RETURN COUNT(n) AS total_nodes"}', name='run_cypher_in_neo4j')

Invoking Recommended Function call:
Function(arguments='{"cypher_query": "MATCH ()-[r]->() RETURN COUNT(r) AS total_relationships"}', name='run_cypher_in_neo4j')

Invoking Recommended Function call:
Function(arguments='{"cypher_query": "MATCH (n) RETURN DISTINCT labels(n) AS node_labels"}', name='run_cypher_in_neo4j')

Invoking Recommended Function call:
Function(arguments='{"cypher_query": "MATCH ()-[r]->() RETURN DISTINCT type(r) AS relationship_types"}', name='run_cypher_in_neo4j')

Output of function call of run_cypher_in_neo4j with cypher_query=MATCH (n) RETURN COUNT(n) AS total_nodes:
|   total_nodes |
|--------------:|
|           883 |

Output of function call of run_cypher_in_neo4j with cypher_query=MATCH ()-[r]->() RETURN COUNT(r) AS total_relationships:
|   total_relationships |
|----------------------:|
|                  2530 |

### Database Analysis

#### Total Counts
- **Total Nodes:** 883
- **Total Relationships:** 2530

#### Node Labels
The database contains the following node labels:
- `Chunk`
- `__Entity__`, `CANDIDATE`
- `__Entity__`, `SOFT_SKILL`
- `__Entity__`, `SOFT_SKILL`, `HARD_SKILL`
- `__Entity__`, `HARD_SKILL`
- `__Entity__`, `PROFESSION`
- `__Entity__`, `EDUCATION`
- `__Entity__`, `JOB_DESCRIPTION`
- `__Entity__`, `EDUCATION`, `COURSE`
- `__Entity__`, `ORGANIZATION`
- `__Entity__`, `PROJECT`
- `__Entity__`, `COURSE`
- `__Entity__`, `HARD_SKILL`, `PROFESSION`
- `__Entity__`, `HARD_SKILL`, `EDUCATION`
- `__Entity__`, `REFERENCE`

#### Relationship Types
The database contains the following relationship types:
- `MENTIONS`
- `HAS_SOFT_SKILL`
- `HAS_HARD_SKILL`
- `HAS_PROFESSION`
- `HAS_EDUCATION`
- `REQUIRES_SKILL`
- `COMPLETED_COURSE`
- `ATTENDED_INSTITUTION`
- `WORKED_ON`
- `HAS_RESPONSIBILITY`
- `WORKS_AT`
- `HAS_REFERENCE`
- `REQUIRES_EDUCATION`

This analysis provides a high-level overview of the structure and composition of the Neo4j database.

In [ ]:
# Trying to call Graph Data science funtions
next_messages = [next_messages[0]]
next_messages.append({
        "role": "user",
        "content": "generate an analysis of the whole graph, like shortest path analysis, centrality analysis and community detection",
    })

assistant_response = run_multiturn_conversation(
    next_messages, tools, available_functions
)
print("Final Response:")
Markdown(next_messages[-1]['content'])

Invoking Recommended Function call:
Function(arguments='{"cypher_query": "MATCH (start:CANDIDATE), (end:JOB_DESCRIPTION)\\nCALL gds.shortestPath.dijkstra.stream({\\n  nodeProjection: \'*\',\\n  relationshipProjection: {\\n    all: {\\n      type: \'*\',\\n      properties: \'weight\'\\n    }\\n  },\\n  startNode: start,\\n  endNode: end,\\n  relationshipWeightProperty: \'weight\'\\n})\\nYIELD index, sourceNode, targetNode, totalCost, nodeIds, costs, path\\nRETURN\\n  index,\\n  gds.util.asNode(sourceNode).name AS sourceNodeName,\\n  gds.util.asNode(targetNode).name AS targetNodeName,\\n  totalCost,\\n  [nodeId IN nodeIds | gds.util.asNode(nodeId).name] AS nodeNames,\\n  costs,\\n  nodes(path) AS path"}', name='run_cypher_in_neo4j')

Invoking Recommended Function call:
Function(arguments='{"cypher_query": "CALL gds.degree.stream({\\n  nodeProjection: \'*\',\\n  relationshipProjection: \'*\'\\n})\\nYIELD nodeId, score\\nRETURN gds.util.asNode(nodeId).name AS nodeName, score\\nORDER BY sc

ClientError: [Statement.SyntaxError] Type mismatch: expected String but was Map (line 2, column 39 (offset: 85))
"CALL gds.shortestPath.dijkstra.stream({"
                                       ^

In [ ]:
!pip install llama-index -q
!pip install llama-index-llms-azure-openai -q
!pip install llama-index-embeddings-azure-openai -q
!pip install llama-index-vector-stores-neo4jvector -q
!pip install llama-index-graph-stores-neo4j -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.5/15.5 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.8/295.8 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.4/79.4 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.2/173.2 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.8/111.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.6/293.6 kB 10.3 MB/s eta 0:00:00


In [ ]:
from llama_index.llms.azure_openai import AzureOpenAI
from llama_index.embeddings.azure_openai import AzureOpenAIEmbedding
from google.colab import userdata
from llama_index.core import Settings

llm = AzureOpenAI(
    engine="gpt-4o",
    deployment_name=userdata.get("AZURE_MODEL_NAME"), # AOI
    api_key=userdata.get("AZURE_API_KEY"),
    azure_endpoint=userdata.get("AZURE_BASE_URL"),
    api_version=userdata.get("AZURE_API_VERSION"),
    temperature=0.0001,
)

embed_model = AzureOpenAIEmbedding(
    model="text-embedding-3-small",
    deployment_name=userdata.get("AZURE_EMBEDDING_NAME"), # AOI
    api_key=userdata.get("AZURE_API_KEY"),
    azure_endpoint=userdata.get("AZURE_BASE_URL"),
    api_version=userdata.get("AZURE_API_VERSION"),
)

In [ ]:
from llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore
from llama_index.core import PropertyGraphIndex

graph_store = Neo4jPropertyGraphStore(
    username="",
    password="",
    url="bolt://localhost:7687",
)

index = PropertyGraphIndex.from_existing(
    llm=llm,
    embed_model=embed_model,
    property_graph_store=graph_store,
    show_progress=False,
)

In [ ]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
# index.as_retriever().retrieve("Which skills exists?")

In [ ]:
print(index.as_retriever(include_text=False).get_prompts())

{}


In [ ]:
nodes_relations = index.as_retriever(include_text=True,similarity_top_k=1).retrieve("Which skills exists?")

for node in nodes_relations[:]:
    # display(node)
    # print()
    # print(node)
    # print()
    # print(node.node_id)
    # print("---")
    # print(node.node_id)
    # print(node.id_)
    # print(node.to_dict()['node'])
    # print(node.to_dict()['node']['relationships'])
    print(node.to_dict()['node']['relationships']['1']['node_id'])
    # print(node.to_dict()['node']['relationships']['1']['node_id'])

similar_chunk_ids = [node.to_dict()['node']['relationships']['1']['node_id'] for node in nodes_relations]

096bc02b-a0cb-48ec-abc2-ad392db625e6
4a624db9-cf47-4688-b699-137f769d6396
72437df9-72b0-467d-825e-3928e17a6f7b
9548106c-0b02-4e49-b263-916e4d74fb53
1ea48206-d6bd-4184-89c2-bd8de37b8b52


In [ ]:
result = run_cypher_in_neo4j(f"""MATCH (n:Chunk)
WHERE n.document_id in {similar_chunk_ids}
RETURN DISTINCT n.text AS context
""")
print(result)

| context                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [ ]:
def get_context_to_answer_user_question(user_question: str) -> str:
    nodes_relations = index.as_retriever(include_text=True,similarity_top_k=10).retrieve(user_question)
    similar_chunk_ids = [node.to_dict()['node']['relationships']['1']['node_id'] for node in nodes_relations]
    context = run_cypher_in_neo4j(f"""MATCH (n:Chunk)
    WHERE n.document_id in {similar_chunk_ids}
    RETURN DISTINCT n.text AS context
    """)
    return context

In [ ]:
tool_user_context = {
    "type": "function",
    "function": {
        "name": "get_context_to_answer_user_question",
        "description": "Retrieve context according to user question in order to reply to them.",
        "parameters": {
            "type": "object",
            "properties": {
                "user_question": {
                    "type": "string",
                    "description": "Question from the chat",
                },
            },
            "required": ["user_question"],
        },
    },
}


In [ ]:
tools = [
    tool_user_context,
]

available_functions = {
    "get_context_to_answer_user_question": get_context_to_answer_user_question,
}

In [ ]:
#11seconds
next_messages = [next_messages[0]]
next_messages.append({
        "role": "user",
        "content": "Tell me the summary resume of ROHITH KANNAN A",
    })

assistant_response = run_multiturn_conversation(
    next_messages, tools, available_functions
)
print("Final Response:")
Markdown(next_messages[-1]['content'])

Invoking Recommended Function call:
Function(arguments='{"user_question":"summary resume of ROHITH KANNAN A"}', name='get_context_to_answer_user_question')

Output of function call of get_context_to_answer_user_question with user_question=summary resume of ROHITH KANNAN A:
| context                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             

### Summary Resume of ROHITH KANNAN A

**Full Name:** ROHITH KANNAN A  
**Email:** rohithkannan.anakkal@gmail.com  

#### Summary
Seeking a challenging opportunity as an engineer to utilize technical and communication skills, gain exposure to latest technologies, and contribute to the success of the organization.

#### Soft Skills
- Communication
- Problem-solving
- Teamwork
- Attention to detail
- Organizational skills

#### Hard Skills
- Electrical engineering
- Electronics engineering
- MATLAB
- Civil CAD
- Electrical CAD
- MS-WORD
- MS-POWER POINT
- MS EXCEL
- PLC
- VFD
- SCADA
- Electrical Control Panel wiring
- Relay Logic
- HMI
- JAVA
- C
- Python ver. 3.98

#### Education
- **Bachelor of Technology in Electrical & Electronics Engineering**  
  Institute: Kannur University, Kerala
- **12th standard**  
  Institute: Durga Higher Secondary School
- **10th standard**  
  Institute: Durga Higher Secondary School

#### Work Experience
- **Electrical Engineer Maintenance**  
  **Duration:** 3rd September 2013 to 28th May 2015 (21 months)  
  **Responsibilities:** Worked in different phases of the project such as designing, maintenance, supervision of electrical works in construction sites, troubleshooting problems regarding electrical circuits, preparation of single line diagrams, layouts, cable schedules, load lists, etc., and preparation of lighting, earthing & hazardous area classification layouts.

- **Customer Service Officer**  
  **Duration:** 24th June 2015 to 12th May 2016 (11 months)  
  **Responsibilities:** Made appointments with customers, collected records from customers, recorded and uploaded data to the database, and retrieved data according to the customer’s demand.

#### References
- None provided

#### CV Path
- [505.pdf](#)

In [ ]:
#10seconds
next_messages = [next_messages[0]]
next_messages.append({
        "role": "user",
        "content": "Tell me the summary resume of Rajinder Pal",
    })

assistant_response = run_multiturn_conversation(
    next_messages, tools, available_functions
)
print("Final Response:")
Markdown(next_messages[-1]['content'])

Invoking Recommended Function call:
Function(arguments='{"user_question":"summary resume of Rajinder Pal"}', name='get_context_to_answer_user_question')

Output of function call of get_context_to_answer_user_question with user_question=summary resume of Rajinder Pal:
| context                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

Here is the summary resume of Rajinder Pal:

**Full Name:** Rajinder Pal  
**Email:** rajwinder.kohli27@gmail.com

### Summary
- No summary provided.

### Soft Skills
- Communication
- Team Management
- Problem Solving
- Organizational Skills
- Customer Service

### Hard Skills
- Transport Management
- Vehicle Maintenance
- Project Management
- Human Resources
- Computer Skills
- Python ver. 3.56

### Education
1. **Matriculation**
   - **Institute/University:** Haryana Board, Bhiwani
   - **Date:** Not provided

2. **10+2**
   - **Institute/University:** Haryana Board, Bhiwani
   - **Date:** Not provided

3. **Graduation**
   - **Institute/University:** Kurukshetra University, Kurukshetra
   - **Date:** Not provided

### Work Experience
1. **Site Transport & Workshop In-charge**
   - **Time Frame:** Aug 2015 to May 2016
   - **Duration:** 9 months
   - **Additional Info:** In charge of the day-to-day operations of the transport department.

2. **Site Transport & Workshop Supervisor**
   - **Time Frame:** Jan 2015 to April 2015
   - **Duration:** 4 months
   - **Additional Info:** Ensured the efficient and effective use of staff, workshop/body-shop facilities, and equipment.

3. **Transport Analyst**
   - **Time Frame:** July 2010 to May 2012
   - **Duration:** 22 months
   - **Additional Info:** Approved time sheets for delivery.

4. **H.R. Coordinator**
   - **Time Frame:** Oct 2007 to Dec 2009
   - **Duration:** 26 months
   - **Additional Info:** Maintained the department office area in an organized and professional manner.

5. **Computer Operator**
   - **Time Frame:** 7th January 2006 to October 2007
   - **Duration:** 22 months
   - **Additional Info:** Made voter cards for every voter in Nawanshahr District on behalf of the Election Commission of India.

6. **Computer Operator**
   - **Time Frame:** Dec 2001 to Aug 2005
   - **Duration:** 45 months
   - **Additional Info:** Generated invoices through specific software modified by the Department of vehicle sold by Canteen store department to the army personnel.

### References
- Steve Sutcliff
- Nigel Emms
- David Allison
- Harish Joshi
- Mr. Paul Krishanmoorthy

### CV Path
- [2470.pdf](#)

This summary provides an overview of Rajinder Pal's skills, education, work experience, and references.

In [ ]:
#10seconds
next_messages = [next_messages[0]]
next_messages.append({
        "role": "user",
        "content": "list job description of Java developer",
    })

assistant_response = run_multiturn_conversation(
    next_messages, tools, available_functions
)
print("Final Response:")
Markdown(next_messages[-1]['content'])

Invoking Recommended Function call:
Function(arguments='{"user_question":"list job description of Java developer"}', name='get_context_to_answer_user_question')

Output of function call of get_context_to_answer_user_question with user_question=list job description of Java developer:
| context                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

Here is a job description for a Full Stack Java Developer:

### Job Title:
Full Stack Java Developer

### Responsibilities:
- Develop software solutions by analyzing system performance standards.
- Confer with users or system engineers to analyze systems flow, data usage, and work processes.
- Investigate problem areas and integrate existing software into new or modified systems.
- Develop and execute test procedures for software components.
- Analyze user requirements to derive technical software design and performance requirements.
- Plan, track, and manage deliverables on short-term sprints and long-term software deployments.

### Required Skills:
- **Education**: Bachelor's Degree preferably in Computer Science, Engineering, Management Information Systems, or equivalent work experience.
- **Experience**: 
  - 3+ years of full stack software development on web and client-server solutions.
  - 3+ years of experience with Java, Angular, ReactJS, CSS, JavaScript, HTML.
  - 3+ years of experience developing web applications utilizing Java Enterprise Edition (J2EE), XML, or Web Services at an enterprise level.
  - Experience with Spring Framework, JMS, SOAP, and REST web services.
  - Understanding of developing Web-services, HTML UI, CSS, and JavaScript.
  - Experience with AWS Cloud, including CI/CD and Kubernetes.
- **Other Requirements**:
  - Ability to travel 10% on average, based on the work you do, the clients you serve, and industries/sectors you serve.
  - Must live near the Lake Mary, FL; Gilbert, AZ; or Mechanicsburg, PA areas.
  - Must be legally authorized to work in the United States without the need for employer sponsorship, now or at any time in the future.

### Preferred Skills:
- Experience in the consulting industry, particularly in areas such as management consulting, strategy consulting, or operations consulting.
- Knowledge of public administration, including familiarity with government operations, public services, or public policy.
- Proven ability to work collaboratively in a multi-disciplinary team environment.

### Summary:
A Full Stack Java Developer will be responsible for designing, developing, and modifying software systems and custom components using modern development stacks. They will develop software solutions by analyzing system performance standards, conferring with users or system engineers, and analyzing systems flow, data usage, and work processes; and investigate problem areas.

In [ ]:
#20seconds
next_messages = [next_messages[0]]
next_messages.append({
        "role": "user",
        "content": "list the best match candidates sorted by importance(maximum 3) to fit the job description Java developer, and tell me why according to prefered and required skills of the job description so meantion they first",
    })

assistant_response = run_multiturn_conversation(
    next_messages, tools, available_functions
)
print("Final Response:")
Markdown(next_messages[-1]['content'])

Invoking Recommended Function call:
Function(arguments='{"user_question":"list the best match candidates sorted by importance(maximum 3) to fit the job description Java developer, and tell me why according to prefered and required skills of the job description so meantion they first"}', name='get_context_to_answer_user_question')

Output of function call of get_context_to_answer_user_question with user_question=list the best match candidates sorted by importance(maximum 3) to fit the job description Java developer, and tell me why according to prefered and required skills of the job description so meantion they first:
| context                                                                                                                                                                                                                                                                                                                                                                             

Based on the job description for a Full Stack Java Developer, the required and preferred skills are as follows:

### Required Skills:
1. **Education**: Bachelor's Degree preferably in Computer Science, Engineering, Management Information Systems, or equivalent work experience.
2. **Experience**:
   - 3+ years of full stack software development on web and client-server solutions.
   - 3+ years of experience with Java, Angular, ReactJS, CSS, JavaScript, HTML.
   - 3+ years of experience developing web applications utilizing Java Enterprise Edition (J2EE), XML, or Web Services at an enterprise level.
   - Experience with Spring Framework, JMS, SOAP, and REST web services.
   - Understanding of developing Web-services, HTML UI, CSS, and JavaScript.
   - Experience with AWS Cloud, including CI/CD and Kubernetes.
3. **Other Requirements**:
   - Ability to travel 10% on average.
   - Must live near Lake Mary, FL; Gilbert, AZ; or Mechanicsburg, PA areas.
   - Must be legally authorized to work in the United States without the need for employer sponsorship.

### Preferred Skills:
1. Experience in the consulting industry, particularly in areas such as management consulting, strategy consulting, or operations consulting.
2. Knowledge of public administration, including familiarity with government operations, public services, or public policy.
3. Proven ability to work collaboratively in a multi-disciplinary team environment.

### Top 3 Candidates:

#### 1. **D.SREEJA**
- **Hard Skills**: C, C++, JAVA, DBMS, HTML, .Net, Python ver. 3.70
- **Education**: B.Tech (C.S.E)
- **Project**: Communicate Aware Opportunity Routing in Mobile Social Network
- **Why**: D.SREEJA has experience with JAVA and HTML, which are crucial for the job. The B.Tech in Computer Science aligns well with the required educational background. The project experience in mobile social networks indicates a strong understanding of web and client-server solutions.

#### 2. **RITESH SRIVASTAVA**
- **Hard Skills**: IT Related Technologies: Big-Data Technologies; IT-Sun Micro Technologies; ERP Packages; Microsoft Technologies; IT Infrastructure; Web Related Technologies, Python ver. 3.30
- **Education**: MBA/PGDM in Human Resources, B.Tech in Applied Electronics and Instrumentation
- **Project**: Lead-HR, Senior Executive - HR, Senior HR Executive, System Support
- **Why**: RITESH has a strong background in IT-related technologies and web-related technologies, which are essential for the role. The B.Tech degree provides a solid technical foundation. Although his recent experience is in HR, his technical skills and educational background make him a strong candidate.

#### 3. **STEPHEN KARANJA NJOROGE**
- **Hard Skills**: QuickBooks, pastel, Ms access Ms word, Ms Excel, Ms PowerPoint, Email and Internet, Branch power and Bank Fusion Universal Banking(BFUB), Python ver. 3.86
- **Education**: Bachelor of Commerce Degree (Accounting Option)
- **Project**: Customer Experience Executive, Relationship Officer - Retail and Business Banking Division, Credit Administration, monitoring and Control, Credit Analyst, Direct Sales Representative
- **Why**: Stephen has a strong analytical background and experience with various software tools. Although his primary education is in commerce, his technical skills in Python and other software tools indicate a capability to adapt and learn new technologies, which is valuable for a Full Stack Java Developer role.

These candidates are selected based on their alignment with the required and preferred skills for the Full Stack Java Developer position.

https://docs.llamaindex.ai/en/latest/module_guides/indexing/lpg_index_guide/

# Combianed both tools

In [ ]:
!nj/bin/neo4j start

Directories in use:
home:         /content/nj
config:       /content/nj/conf
logs:         /content/nj/logs
plugins:      /content/nj/plugins
import:       /content/nj/import
data:         /content/nj/data
certificates: /content/nj/certificates
licenses:     /content/nj/licenses
run:          /content/nj/run
Starting Neo4j.
Started neo4j (pid:17624). It is available at http://localhost:7474
There may be a short delay until the server is ready.


In [ ]:
from IPython.display import Markdown
from py2neo import Graph
graph = Graph("bolt://localhost:7687")

In [ ]:
from llama_index.graph_stores.neo4j import Neo4jPropertyGraphStore
from llama_index.core import PropertyGraphIndex

graph_store = Neo4jPropertyGraphStore(
    username="",
    password="",
    url="bolt://localhost:7687",
)


index = PropertyGraphIndex.from_existing(
    llm=llm,
    embed_model=embed_model,
    property_graph_store=graph_store,
    show_progress=False,
)

In [ ]:
def run_cypher_in_neo4j(cypher_query: str, df=False) -> str:
    result = graph.run(cypher_query).to_data_frame()
    return result if df else result.to_markdown(index=False)

def get_context_to_answer_user_question(user_question: str) -> str:
    nodes_relations = index.as_retriever(include_text=True,similarity_top_k=10).retrieve(user_question)
    similar_chunk_ids = [node.to_dict()['node']['relationships']['1']['node_id'] for node in nodes_relations]
    context = run_cypher_in_neo4j(f"""MATCH (n:Chunk)
    WHERE n.document_id in {similar_chunk_ids}
    RETURN DISTINCT n.text AS context
    """)
    return context


available_functions = {
    "run_cypher_in_neo4j": run_cypher_in_neo4j,
    "get_context_to_answer_user_question": get_context_to_answer_user_question,
}

In [ ]:
#@title run_multiturn_conversation {display-mode: "form"}
from openai import AzureOpenAI
from google.colab import userdata
import json
from pprint import pprint

client = AzureOpenAI(
    api_key=userdata.get('AZURE_API_KEY'),
    api_version=userdata.get('AZURE_API_VERSION'),
    azure_endpoint=userdata.get('AZURE_BASE_URL')
)

import concurrent.futures

BLUE,GREEN,MANGETA,RESET = '\033[94m', '\033[92m', '\033[95m', '\033[0m'
def dict_to_expression_string(dictionary):
    return ' '.join(f'{BLUE}{key}{RESET}={GREEN}{value}{RESET}' for key, value in dictionary.items())

def run_multiturn_conversation(messages, tools, available_functions):
    response = client.chat.completions.create(
        messages=messages,
        tools=tools,
        tool_choice="auto",
        model=userdata.get('AZURE_MODEL_NAME'),
        temperature=1e-4,
    )

    while response.choices[0].finish_reason == "tool_calls":
        response_message = response.choices[0].message
        with concurrent.futures.ThreadPoolExecutor() as executor:
            futures, names, args = [], [], []
            for tool in response_message.tool_calls:
                print("Invoking Recommended Function call:")
                print(tool.function)
                print()
                function_name = tool.function.name
                names.append(function_name)
                function_to_call = available_functions[function_name]
                function_args = json.loads(tool.function.arguments)
                args.append(function_args)
                futures.append(executor.submit(function_to_call,**function_args))

        for future, name, arg in zip(futures, names, args):
            function_response = future.result()
            print(f"Output of function call of {MANGETA}{name}{RESET} with {dict_to_expression_string(arg)}:")
            print(function_response)
            print()
            messages.append({
                    "role": response_message.role,
                    "function_call": {
                        "name": tool.function.name,
                        "arguments": tool.function.arguments,
                    },
                    "content": None,
                })
            messages.append({
                    "role": "function",
                    "name": function_name,
                    "content": function_response,
                })
        print("Messages in next request:")
        for message in messages:
            print(message)
        print()
        response = client.chat.completions.create(
            messages=messages,
            tools=tools,
            tool_choice="auto",
            model=userdata.get('AZURE_MODEL_NAME'),
            temperature=1e-4,
        )
    messages.append({
            "role": response.choices[0].message.role,
            "content": response.choices[0].message.content,
        })
    return response

In [ ]:
run_cypher_in_neo4j("""
MATCH (c:CANDIDATE)-[:HAS_SOFT_SKILL]->(s:SOFT_SKILL)
WITH c, s, apoc.text.sorensenDiceSimilarity(c.name, ' PRASANNA') AS similarity
WHERE similarity > 0.7
RETURN c.name AS candidate_name, s.name AS soft_skill, similarity
""",df=True)

,candidate_name,soft_skill,similarity
0,G S PRASANNA KUMAR,Team work and team building,0.777778
1,G S PRASANNA KUMAR,Strong leadership skills,0.777778
2,G S PRASANNA KUMAR,Strong Motivation,0.777778
3,G S PRASANNA KUMAR,Communication,0.777778
4,G S PRASANNA KUMAR,Problem-solving,0.777778


In [ ]:
run_cypher_in_neo4j("""
WITH ['J. MADISON ILER', 'Aslam KHAN 123', 'g s PRASANA KUM'] AS candidateNames
MATCH (c:CANDIDATE)-[:HAS_SOFT_SKILL]->(s:SOFT_SKILL)
UNWIND candidateNames AS givenName
WITH c, s, givenName, apoc.text.sorensenDiceSimilarity(c.name, givenName) AS similarity
WHERE similarity > 0.8
RETURN c.name AS candidate_name, s.name AS soft_skill, similarity

""",df=True)

,candidate_name,soft_skill,similarity
0,G S PRASANNA KUMAR,Team work and team building,0.842105
1,G S PRASANNA KUMAR,Strong leadership skills,0.842105
2,G S PRASANNA KUMAR,Strong Motivation,0.842105
3,G S PRASANNA KUMAR,Communication,0.842105
4,G S PRASANNA KUMAR,Problem-solving,0.842105
5,J. MADISON ILER,Team work and team building,1.000000
6,J. MADISON ILER,Strong analytical skills,1.000000
7,J. MADISON ILER,Communication,1.000000
8,J. MADISON ILER,Customer service,1.000000
9,J. MADISON ILER,Problem-solving,1.000000


In [ ]:
run_cypher_in_neo4j("""
CALL apoc.help('apoc.text') YIELD name, text
RETURN name, text
""",df=True)

,name,text
0,apoc.text.phoneticDelta,Returns the US_ENGLISH soundex character diffe...
1,apoc.text.base64Decode,Decodes the given Base64 encoded `STRING`.
2,apoc.text.base64Encode,Encodes the given `STRING` with Base64.
3,apoc.text.base64UrlDecode,Decodes the given Base64 encoded URL.
4,apoc.text.base64UrlEncode,Encodes the given URL with Base64.
5,apoc.text.byteCount,Returns the size of the given `STRING` in bytes.
6,apoc.text.bytes,Returns the given `STRING` as bytes.
7,apoc.text.camelCase,Converts the given `STRING` to camel case.
8,apoc.text.capitalize,Capitalizes the first letter of the given `STR...
9,apoc.text.capitalizeAll,Capitalizes the first letter of every word in ...


In [ ]:
tools =  [
        {
            "type": "function",
            "function": {
                "name": "run_cypher_in_neo4j",
                "description": "Execute a Cypher query in a Neo4j database and return the result as a Markdown string. Use this tool to answer mathematical operations, such as counts, measures, list names, and more.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "cypher_query": {
                            "type": "string",
                            "description":
"""The Cypher query to execute.
This query should always use complex aliases to identify the query result and use name field explicity in the result.
This query should always use apoc.text.sorensenDiceSimilarity to compare two strings and match with a score higher than 0.7 instead of using a simple name comparation(i.e. node.name='...')
"""
                        }
                    },
                    "required": ["cypher_query"]
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "get_context_to_answer_user_question",
                "description": "Retrieve context according to the user's question to provide an accurate response. Use this tool to answer specific information requests, such as descriptions, job description info, matches, summaries, and more.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "user_question": {
                            "type": "string",
                            "description":
"""The question asked by the user in the chat."""
                        }
                    },
                    "required": ["user_question"]
                }
            }
        }
    ]

In [ ]:
nodes_and_relations = run_cypher_in_neo4j("""CALL db.labels() YIELD label
RETURN "Node" AS Type, label AS Name
UNION ALL
CALL db.relationshipTypes() YIELD relationshipType
RETURN "Relationship" AS Type, relationshipType AS Name""")
node_relations_map = run_cypher_in_neo4j("""MATCH (n)-[r]->()
WITH labels(n) AS NodeLabels, collect(DISTINCT type(r)) AS Relationships
RETURN DISTINCT NodeLabels[-1] AS Node, Relationships
""")

In [ ]:
system_prompt = f"""
Assistant is a helpful assistant that helps users get answers to questions about a Candidates, resumes, job descriptions, matches, and so on.
Assistant has access to several tools and sometimes you may need to call multiple tools in sequence or combinated them to get answers for your users.

Check carefully your log/history perphaps the question was replied, and always use the essential number of tools.

The Neo4j graph schema, including all entities and relationships available, is as follows (use the exact names):
{nodes_and_relations}

Here is the map of nodes and relationships:
{node_relations_map}
"""
next_messages = [
    {
        "role": "system",
        "content": system_prompt,
    }]

In [ ]:
next_messages = [next_messages[0]]
next_messages.append({
        "role": "user",
        "content": "List soft skills of J. MADISON ILER, Aslam Khan, g s PRASANNA KUMAR", # 9 seconds
    })

assistant_response = run_multiturn_conversation(
    next_messages, tools, available_functions
)
print("Final Response:")
Markdown(next_messages[-1]['content'])

Invoking Recommended Function call:
Function(arguments='{"cypher_query": "MATCH (c:CANDIDATE)-[:HAS_SOFT_SKILL]->(s:SOFT_SKILL) WHERE apoc.text.sorensenDiceSimilarity(c.name, \'J. MADISON ILER\') > 0.7 RETURN c.name AS candidate_name, s.name AS soft_skill"}', name='run_cypher_in_neo4j')

Invoking Recommended Function call:
Function(arguments='{"cypher_query": "MATCH (c:CANDIDATE)-[:HAS_SOFT_SKILL]->(s:SOFT_SKILL) WHERE apoc.text.sorensenDiceSimilarity(c.name, \'Aslam Khan\') > 0.7 RETURN c.name AS candidate_name, s.name AS soft_skill"}', name='run_cypher_in_neo4j')

Invoking Recommended Function call:
Function(arguments='{"cypher_query": "MATCH (c:CANDIDATE)-[:HAS_SOFT_SKILL]->(s:SOFT_SKILL) WHERE apoc.text.sorensenDiceSimilarity(c.name, \'g s PRASANNA KUMAR\') > 0.7 RETURN c.name AS candidate_name, s.name AS soft_skill"}', name='run_cypher_in_neo4j')

Output of function call of run_cypher_in_neo4j with cypher_query=MATCH (c:CANDIDATE)-[:HAS_SOFT_SKILL]->(s:SOFT_SKILL) WHERE apoc.text.

Here are the soft skills for each candidate:

### J. MADISON ILER
- Team work and team building
- Strong analytical skills
- Communication
- Customer service
- Problem-solving
- Attention to detail

### Aslam Khan
- Team work and team building
- Communication
- Problem-solving

### G S PRASANNA KUMAR
- Team work and team building
- Strong leadership skills
- Strong Motivation
- Communication
- Problem-solving

In [ ]:
next_messages = [next_messages[0]]
next_messages.append({
        "role": "user",
        "content": "list the best match candidates sorted by importance(maximum 3) to fit the job description Full stock Java developer, and tell me why match or not according to prefered and required skills of the job description so mention they first", # 100 seconds
    })

assistant_response = run_multiturn_conversation(
    next_messages, tools, available_functions
)
print("Final Response:")
Markdown(next_messages[-1]['content'])

Invoking Recommended Function call:
Function(arguments='{"user_question":"list the best match candidates sorted by importance(maximum 3) to fit the job description Full stock Java developer, and tell me why match or not according to prefered and required skills of the job description so mention they first"}', name='get_context_to_answer_user_question')

Output of function call of get_context_to_answer_user_question with user_question=list the best match candidates sorted by importance(maximum 3) to fit the job description Full stock Java developer, and tell me why match or not according to prefered and required skills of the job description so mention they first:
| context                                                                                                                                                                                                                                                                                                                               

Based on the job description for a Full Stack Java Developer, here are the top 3 candidates who best match the required and preferred skills:

### 1. **D.SREEJA**
- **Email:** diddisreeja69@gmail.com
- **Skills:**
  - **Hard Skills:** C, C++, JAVA, DBMS, HTML, .Net, Python ver. 3.70
  - **Education:** B.Tech (C.S.E) from J.N.T.U Hyderabad (Ganapathy engineering college, Warangal)
- **Why Match:**
  - **Required Skills:** D.SREEJA has experience with JAVA, HTML, and DBMS, which aligns with the required skills of Java, HTML, and database management.
  - **Education:** Holds a B.Tech in Computer Science and Engineering, which matches the preferred educational background.
  - **Additional Skills:** Knowledge of C, C++, and .Net adds value to her profile.

### 2. **HAMEED SATHIK M**
- **Email:** hameedsathik@gmail.com
- **Skills:**
  - **Hard Skills:** Proficient in MS Office, AutoCAD, Small world 2.1, Ground V10 & CMD software, Basics in C, C++ and JAVA, Python ver. 3.112
  - **Education:** Bachelor of Engineering from Mohamed Sathak Engineering College, Kilakarai
- **Why Match:**
  - **Required Skills:** HAMEED SATHIK M has experience with JAVA and Python, which aligns with the required skills of Java and additional programming languages.
  - **Education:** Holds a Bachelor of Engineering, which matches the preferred educational background.
  - **Additional Skills:** Proficiency in AutoCAD and other software tools adds value to his profile.

### 3. **RITESH SRIVASTAVA**
- **Email:** riteshforjobs@gmail.com
- **Skills:**
  - **Hard Skills:** IT Related Technologies: Big-Data Technologies; IT-Sun Micro Technologies; ERP Packages; Microsoft Technologies; IT Infrastructure; Web Related Technologies, Python ver. 3.30
  - **Education:** MBA/PGDM in Human Resources from Birla Institute of Management Technology (BIMTECH), Greater Noida; B.Tech in Applied Electronics and Instrumentation from Uttar Pradesh Technical University, Lucknow
- **Why Match:**
  - **Required Skills:** RITESH SRIVASTAVA has experience with various IT-related technologies and Python, which aligns with the required skills of programming languages and frameworks.
  - **Education:** Holds a B.Tech in Applied Electronics and Instrumentation, which is relevant to the preferred educational background.
  - **Additional Skills:** Experience in HR and talent acquisition adds value to his profile.

These candidates have been selected based on their alignment with the required and preferred skills for the Full Stack Java Developer position.

In [ ]:
next_messages = [next_messages[0]]
next_messages.append({
        "role": "user",
        "content": "How many jobs description, candidates, skills exist?", # 9 seconds
    })

assistant_response = run_multiturn_conversation(
    next_messages, tools, available_functions
)
print("Final Response:")
Markdown(next_messages[-1]['content'])

Invoking Recommended Function call:
Function(arguments='{"cypher_query": "MATCH (n:JOB_DESCRIPTION) RETURN COUNT(n) AS job_description_count"}', name='run_cypher_in_neo4j')

Invoking Recommended Function call:
Function(arguments='{"cypher_query": "MATCH (n:CANDIDATE) RETURN COUNT(n) AS candidate_count"}', name='run_cypher_in_neo4j')

Invoking Recommended Function call:
Function(arguments='{"cypher_query": "MATCH (n:SOFT_SKILL) RETURN COUNT(n) AS soft_skill_count"}', name='run_cypher_in_neo4j')

Invoking Recommended Function call:
Function(arguments='{"cypher_query": "MATCH (n:HARD_SKILL) RETURN COUNT(n) AS hard_skill_count"}', name='run_cypher_in_neo4j')

Output of function call of run_cypher_in_neo4j with cypher_query=MATCH (n:JOB_DESCRIPTION) RETURN COUNT(n) AS job_description_count:
|   job_description_count |
|------------------------:|
|                       6 |

Output of function call of run_cypher_in_neo4j with cypher_query=MATCH (n:CANDIDATE) RETURN COUNT(n) AS candidate_coun

Here are the counts for each category:

- **Job Descriptions**: 6
- **Candidates**: 108
- **Soft Skills**: 201
- **Hard Skills**: 301

In [ ]:
next_messages = [next_messages[0]]
next_messages.append({
        "role": "user",
        "content": "How many jobs description exist? and then tell me the name of 2 candidates which best match with Java Developer", # 11 seconds
    })

assistant_response = run_multiturn_conversation(
    next_messages, tools, available_functions
)
print("Final Response:")
Markdown(next_messages[-1]['content'])

Invoking Recommended Function call:
Function(arguments='{"cypher_query": "MATCH (j:JOB_DESCRIPTION) RETURN count(j) AS job_count"}', name='run_cypher_in_neo4j')

Invoking Recommended Function call:
Function(arguments='{"user_question": "name of 2 candidates which best match with Java Developer"}', name='get_context_to_answer_user_question')

Output of function call of run_cypher_in_neo4j with cypher_query=MATCH (j:JOB_DESCRIPTION) RETURN count(j) AS job_count:
|   job_count |
|------------:|
|           6 |

Output of function call of get_context_to_answer_user_question with user_question=name of 2 candidates which best match with Java Developer:
| context                                                                                                                                                                                                                                                                                                                                                

There are 6 job descriptions in the database.

Based on the context provided, here are two candidates that best match the "Java Developer" role:

1. **HAMEED SATHIK M**
   - **Email:** hameedsathik@gmail.com
   - **Soft Skills:** Problem solving capacity, Strong Communication, Strong Motivation, Team Work & Co-ordination, Able to work in Challenging Job environment
   - **Hard Skills:** Basics in C, C++, and JAVA, Python ver. 3.112
   - **Education:** Bachelor of Engineering from Mohamed Sathak Engineering College, Kilakarai (2010-2014)
   - **Work Experience:** HVAC Technical Coordinator (Sep 2014 - Dec 2017)

2. **Jitendra Kumar Yadav**
   - **Email:** yadavjitendra967@gmail.com
   - **Soft Skills:** Punctual, Confident, Sincere, Responsible, Dedicated, Ability to work under pressure
   - **Hard Skills:** Web Technologies (HTML, CSS, JavaScript, Jquery, Ajax), Python ver. 3.117
   - **Education:** BCA from BSTM, Kolkata (2013)
   - **Work Experience:** Computer Operator at NIHFW Pvt. Ltd. (March 2016 - Present), Eminenture Pvt Ltd. (January 2015 - February 2016), Encoder Pvt Ltd. (August 2013 - September 2014)

These candidates have relevant skills and experience that align with the requirements for a Java Developer role.

In [ ]:
next_messages = [next_messages[0]]
next_messages.append({
        "role": "user",
        "content": "How many candidates exist? and then select 2 random candidates. Then extract their existing summary", # 25 seconds
    })

assistant_response = run_multiturn_conversation(
    next_messages, tools, available_functions
)
print("Final Response:")
Markdown(next_messages[-1]['content'])

Invoking Recommended Function call:
Function(arguments='{"cypher_query": "MATCH (c:CANDIDATE) RETURN count(c) as count"}', name='run_cypher_in_neo4j')

Invoking Recommended Function call:
Function(arguments='{"cypher_query": "MATCH (c:CANDIDATE) RETURN c.name as name ORDER BY rand() LIMIT 2"}', name='run_cypher_in_neo4j')

Output of function call of run_cypher_in_neo4j with cypher_query=MATCH (c:CANDIDATE) RETURN count(c) as count:
|   count |
|--------:|
|     108 |

Output of function call of run_cypher_in_neo4j with cypher_query=MATCH (c:CANDIDATE) RETURN c.name as name ORDER BY rand() LIMIT 2:
| name                   |
|:-----------------------|
| Anitha B               |
| Harshal Maruti Sondkar |

Messages in next request:
{'role': 'system', 'content': "\nAssistant is a helpful assistant that helps users get answers to questions about a Candidates, resumes, job descriptions, matches, and so on.\nAssistant has access to several tools and sometimes you may need to call multiple to

### Summary of Selected Candidates

#### 1. Anitha B
- **Email:** bala14_cse@yahoo.co.in
- **Summary:** Anitha B is an art and craft teacher with a passion for teaching and a strong desire to obtain a part-time job in the field. She has extensive experience working part-time at MCC School and Quaid-e-Millath College, and has conducted workshops at various schools and colleges. Anitha is skilled in conducting art and craft classes, working effectively in a team, and independently. She has a strong knowledge of art and craft, including best out of waste, paper craft, canvas painting, glass painting, fabric painting, and relief art.
- **Soft Skills:** Good knowledge in art and craft, Expertise in conducting art and craft classes, Ability to work effectively in a team as well as independently
- **Hard Skills:** Best out of waste, Paper Craft, Canvas painting, Glass painting, Fabric Painting, Relief Art, etc., Python ver. 3.59
- **Education:** 
  - B.Sc (Computer Science), Institute not mentioned
- **Work Experience:**
  - Trainee Teacher, Art & Craft (2015 – till now)
- **CV Path:** 2507.pdf

#### 2. Harshal Maruti Sondkar
- **Email:** harshalsondkar08@gmail.com
- **Summary:** Chemical engineer with 6+ years of experience in production and process design, seeking to contribute to the progress of the chemical engineering industry.
- **Soft Skills:** Quick learner, Adapts well to changes and pressure in the workplace, Managing relationships & working efficiently with diverse groups of people, Committed to meeting deadlines and schedules, Leadership skills to lead projects & handle work independently
- **Hard Skills:** Process Design Engineering, Aspen Hysys, HTRI, MS Office, Word, Excel, Power Point, Python ver. 3.37
- **Education:**
  - B.E. CHEMICAL, Shivajirao S. Jondhale College of Engg., 2016
  - DIPLOMA, B. L. Patil Polytechnic, Khopoli, 2013
  - Process Design Engineering (PG), Altitude Institute, Mumbai, 2017
- **Work Experience:**
  - Production Supervisor, July 2016 – Present
  - Process Plant Operator, July 2016 – July 2017
  - Industrial training, May and June 2011-2012
- **CV Path:** 1904.pdf

In [ ]:
next_messages = [next_messages[0]]
next_messages.append({
        "role": "user",
        "content": "Tell me one random candidate summary, then list 3 candidates with similar/ankin soft skills", # 9 seconds
    })

assistant_response = run_multiturn_conversation(
    next_messages, tools, available_functions
)
print("Final Response:")
Markdown(next_messages[-1]['content'])

Invoking Recommended Function call:
Function(arguments='{"cypher_query":"MATCH (c:CANDIDATE)-[:HAS_SOFT_SKILL]->(s:SOFT_SKILL) WITH c, COLLECT(s.name) AS soft_skills RETURN c.name AS candidate_name, soft_skills ORDER BY RAND() LIMIT 1"}', name='run_cypher_in_neo4j')

Output of function call of run_cypher_in_neo4j with cypher_query=MATCH (c:CANDIDATE)-[:HAS_SOFT_SKILL]->(s:SOFT_SKILL) WITH c, COLLECT(s.name) AS soft_skills RETURN c.name AS candidate_name, soft_skills ORDER BY RAND() LIMIT 1:
| candidate_name        | soft_skills                                                                  |
|:----------------------|:-----------------------------------------------------------------------------|
| Yogesh Chandra Pandey | ['Strong leadership skills', 'Team work and team building', 'Communication'] |

Messages in next request:
{'role': 'system', 'content': "\nAssistant is a helpful assistant that helps users get answers to questions about a Candidates, resumes, job descriptions, matches

### Random Candidate Summary:
**Candidate Name:** Yogesh Chandra Pandey  
**Soft Skills:** 
- Strong leadership skills
- Team work and team building
- Communication

### Candidates with Similar/Ankin Soft Skills:
1. **J. MADISON ILER**
2. **Aslam Khan**
3. **G S PRASANNA KUMAR**